# Análisis Exploratorio : Fallecidos y Lesionados (2018-2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.  
El objetivo es explorar los microdatos oficiales de fallecidos y lesionados en hechos de tránsito proporcionados por el  
Instituto Nacional de Estadística (INE) para el septenio 2018-2024, identificando la estructura,  
calidad y distribución de las variables antes de cualquier transformación.

> **Fuente:** INE — Microdatos de Fallecidos y Lesionados (2018-2024)  

In [1]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de Datos

In [2]:
# -- Carga de datos ----------------------------------------------
RAW_PATH = '../data/raw/ACCIDENTES DE TRANSITO - FALLECIDOS Y LESIONADOS'
YEARS = range(2018, 2025)

frames = []
for year in YEARS:
    path = os.path.join(RAW_PATH, f'fallecidos-y-lesionados-ano-{year}.xlsx')
    df_year = pd.read_excel(path)
    df_year['año_carga'] = year
    frames.append(df_year)
    print(f'  {year}: {len(df_year):,} registros cargados')

df_fl = pd.concat(frames, ignore_index=True)
print(f'\n✓ Total registros: {len(df_fl):,}')

  2018: 9,407 registros cargados
  2019: 10,664 registros cargados
  2020: 8,142 registros cargados
  2021: 10,544 registros cargados
  2022: 10,722 registros cargados
  2023: 11,198 registros cargados
  2024: 11,274 registros cargados

✓ Total registros: 71,951


## 2. Vista General del Dataset

In [3]:
# -- Dimensiones y tipos de dato ----------------------------------------------
print(f'Dimensiones: {df_fl.shape[0]:,} filas × {df_fl.shape[1]} columnas')
print(f'\nColumnas y tipos de dato:')
print(df_fl.dtypes)

Dimensiones: 71,951 filas × 33 columnas

Columnas y tipos de dato:
núm_corre            float64
año_ocu                int64
día_ocu              float64
hora_ocu               int64
g_hora                 int64
g_hora_5               int64
mes_ocu                int64
día_sem_ocu          float64
mupio_ocu              int64
depto_ocu              int64
zona_ocu               int64
sexo_per               int64
edad_per               int64
g_edad_80ymás        float64
g_edad_60ymás        float64
edad_quinquenales      int64
mayor_menor            int64
tipo_veh               int64
marca_veh              int64
color_veh              int64
modelo_veh             int64
g_modelo_veh           int64
tipo_eve               int64
fall_les               int64
int_o_noint            int64
año_carga              int64
Núm_corre            float64
zona_ciudad          float64
num_corre            float64
dia_ocu              float64
dia_sem_ocu          float64
g_edad_80ymas        float64
g_eda

## 2. Vista General

33 columnas totales : mismo patrón de duplicados por año que los datasets anteriores.
Variables nuevas exclusivas: `fall_les` y `int_o_noint`.
Comparación de volumen entre los 3 datasets:

| Dataset | Registros |
|---|---|
| Hechos | 52,488 |
| Fallecidos/Lesionados | 71,951 |
| Vehículos involucrados | 80,721 |

Promedio de 1.37 víctimas por hecho y 1.54 vehículos por hecho.

In [4]:
# -- Columnas por año ----------------------------------------------
for year in YEARS:
    path = os.path.join(RAW_PATH, f'fallecidos-y-lesionados-ano-{year}.xlsx')
    cols = pd.read_excel(path, nrows=0).columns.tolist()
    print(f'\n{year}: {cols}')


2018: ['núm_corre', 'año_ocu', 'día_ocu', 'hora_ocu', 'g_hora', 'g_hora_5', 'mes_ocu', 'día_sem_ocu', 'mupio_ocu', 'depto_ocu', 'zona_ocu', 'sexo_per', 'edad_per', 'g_edad_80ymás', 'g_edad_60ymás', 'edad_quinquenales', 'mayor_menor', 'tipo_veh', 'marca_veh', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve', 'fall_les', 'int_o_noint']

2019: ['núm_corre', 'año_ocu', 'día_ocu', 'hora_ocu', 'g_hora', 'g_hora_5', 'mes_ocu', 'día_sem_ocu', 'depto_ocu', 'mupio_ocu', 'zona_ocu', 'sexo_per', 'edad_per', 'g_edad_80ymás', 'g_edad_60ymás', 'edad_quinquenales', 'mayor_menor', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve', 'fall_les', 'tipo_veh', 'marca_veh', 'int_o_noint']

2020: ['Núm_corre', 'año_ocu', 'día_ocu', 'hora_ocu', 'g_hora', 'g_hora_5', 'mes_ocu', 'día_sem_ocu', 'depto_ocu', 'mupio_ocu', 'zona_ocu', 'sexo_per', 'edad_per', 'g_edad_80ymás', 'g_edad_60ymás', 'edad_quinquenales', 'mayor_menor', 'tipo_veh', 'marca_veh', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve', '

### Hallazgo 1 — Inconsistencias de nombres y orden de columnas

Mismo patrón que datasets anteriores más un hallazgo nuevo:

| Columna estándar | Variantes | Años problemáticos |
|---|---|---|
| `num_corre` | `núm_corre`, `Núm_corre`, `num_corre` | 2018-21, 2020, 2022-24 |
| `dia_ocu` | `día_ocu`, `dia_ocu` | 2018-22 vs 2023-24 |
| `dia_sem_ocu` | `día_sem_ocu`, `dia_sem_ocu` | 2018-22 vs 2023-24 |
| `g_edad_80ymas` | `g_edad_80ymás`, `g_edad_80ymas` | 2018-22 vs 2023-24 |
| `zona_ciudad` | extra | Solo 2021 |

**Hallazgo nuevo ; 2024 invierte el orden de g_hora y g_hora_5:**
- Todos los años: `g_hora` → `g_hora_5`
- 2024: `g_hora_5` → `g_hora`

No afecta los datos pero confirma falta de control de calidad del INE.
`fall_les

In [5]:
# -- Valores ignorados variables críticas + nuevas ----------------------------------------------
IGNORE_CODES = {
    'fall_les'         : 3,
    'int_o_noint'      : 9,
    'sexo_per'         : 9,
    'edad_quinquenales': 18,
    'mayor_menor'      : 9,
    'tipo_veh'         : 99,
    'tipo_eve'         : 99,
    'g_hora_5'         : 4,
    'zona_ocu'         : 99,
    'marca_veh'        : 999,
    'modelo_veh'       : 9999,
}

print(f'{"Variable":<20} {"Ignorados":>10} {"% del total":>12}')
print('-' * 45)
for col, code in IGNORE_CODES.items():
    n = (df_fl[col] == code).sum()
    pct = n / len(df_fl) * 100
    print(f'{col:<20} {n:>10,} {pct:>11.1f}%')

Variable              Ignorados  % del total
---------------------------------------------
fall_les                      0         0.0%
int_o_noint                  44         0.1%
sexo_per                    211         0.3%
edad_quinquenales         6,947         9.7%
mayor_menor               2,137         3.0%
tipo_veh                  2,825         3.9%
tipo_eve                    259         0.4%
g_hora_5                     39         0.1%
zona_ocu                 52,916        73.5%
marca_veh                21,400        29.7%
modelo_veh               49,923        69.4%


### Hallazgo 2 : Calidad de variables

| Variable | Ignorados | % | Decisión |
|---|---|---|---|
| `fall_les` | 0 | **0.0%** | Confiable — variable target |
| `int_o_noint` | 44 | 0.1% | Confiable |
| `sexo_per` | 211 | 0.3% | Confiable |
| `g_hora_5` | 39 | 0.1% | Confiable |
| `tipo_eve` | 259 | 0.4% | Confiable |
| `tipo_veh` | 2,825 | 3.9% | Confiable |
| `mayor_menor` | 2,137 | 3.0% | Confiable |
| `edad_quinquenales` | 6,947 | 9.7% | Aceptable |
| `marca_veh` | 21,400 | 29.7% | Mantener como categoría |
| `modelo_veh` | 49,923 | 69.4% | Descartar |
| `zona_ocu` | 52,916 | 73.5% | Mantener como categoría "No aplica" |

**`fall_les` tiene 0% ignorados : variable target perfecta para el modelo.**  
Este dataset es el más limpio de los tres.

**`zona_ocu` no es un caso de dato ignorado**: en Guatemala capital (municipio 101) solo el 0.5% de los registros tiene `zona_ocu = 99`, mientras que en el resto del país el 90.15% lo tiene. Esto refleja que el sistema de zonas urbanas numeradas solo existe en el municipio de Guatemala y en unas pocas cabeceras municipales — fuera de esas áreas, el código 99 significa que no aplica numeración de zonas, no que el dato se haya perdido.

In [6]:
# -- Distribución de fall_les (variable target) ----------------------------------------------
print(df_fl['fall_les'].value_counts().sort_index())

fall_les
1    13741
2    58201
9        9
Name: count, dtype: int64


### Distribución de fall_les (variable target)

| Código | Categoría | Registros | % |
|---|---|---|---|
| 1 | Fallecido | 13,741 | 19.1% |
| 2 | Lesionado | 58,201 | 80.9% |
| 9 | **Código no documentado** | 9 | 0.0% |

**19.1% de fallecidos : dataset con clases desbalanceadas.**  
Ratio lesionado/fallecido de 4.2:1 , requiere técnicas de balanceo en el modelo.

**Hallazgo nuevo : código 9 en fall_les no está en el diccionario.**  
El diccionario define ignorado como código 3, pero aparecen 9 registros con código 9.  
Son solo 9 casos : se pueden tratar como ignorados o eliminar en limpieza.

In [7]:
# -- Verificar en qué años aparece el código 9 ----------------------------------------------
print(df_fl[df_fl['fall_les'] == 9][['año_carga', 'fall_les']].value_counts())

año_carga  fall_les
2019       9           4
2023       9           2
2024       9           2
2022       9           1
Name: count, dtype: int64


### Hallazgo 3 : Código 9 en fall_les

Aparece en 4 años distintos (2019, 2022, 2023, 2024) , no es error puntual.  
Son 9 registros dispersos sin patrón claro de año.  
**Acción ETL:** eliminar estos 9 registros , no son imputables y  
contaminan la variable target.

In [8]:
# -- Distribución de int_o_noint ----------------------------------------------
print(df_fl['int_o_noint'].value_counts().sort_index())

int_o_noint
1    54290
2    17617
9       44
Name: count, dtype: int64


### Distribución de int_o_noint

| Código | Categoría | Registros | % |
|---|---|---|---|
| 1 | Internado | 54,290 | 75.5% |
| 2 | No internado | 17,617 | 24.5% |
| 9 | Ignorado | 44 | 0.1% |

**75.5% de víctimas fueron internadas** : dato consistente con la gravedad de los hechos.  
Variable muy confiable (0.1% ignorados) : útil como feature secundaria del modelo.

In [9]:
# -- fall_les vs tipo_eve ----------------------------------------------
cross = pd.crosstab(
    df_fl[df_fl['tipo_eve'] != 99]['tipo_eve'],
    df_fl[df_fl['tipo_eve'] != 99]['fall_les'].map({1:'Fallecido', 2:'Lesionado'}),
    normalize='index'
).round(3) * 100

cross.index = ['Colisión','Choque','Vuelco','Caída',
               'Atropello','Derrape','Embarranco','Encuneto']
print(cross.to_string())

fall_les    Fallecido  Lesionado
Colisión         14.0       86.0
Choque           23.9       76.1
Vuelco           13.2       86.8
Caída            23.0       77.0
Atropello        26.5       73.5
Derrape          30.1       69.9
Embarranco       21.6       78.4
Encuneto         13.8       86.2


### fall_les vs tipo_eve : Tasa de mortalidad por tipo de evento

| Tipo | Fallecido | Lesionado |
|---|---|---|
| Colisión | 14.0% | 86.0% |
| Choque | 23.9% | 76.1% |
| Vuelco | 13.2% | 86.8% |
| Caída | 23.0% | 77.0% |
| Atropello | **26.5%** | 73.5% |
| Derrape | **30.1%** | 69.9% |
| Embarranco | 21.6% | 78.4% |
| Encuneto | 13.8% | 86.2% |

**Derrape es el más letal (30.1%)** : coincide con su predominancia en moto.  
**Atropello segundo más letal (26.5%)** : peatón vs vehículo sin protección.  
Colisión y Vuelco son los menos letales (~13–14%) pese a ser los más frecuentes.  
`tipo_eve` es el predictor más fuerte identificado hasta ahora.

In [10]:
# -- fall_les vs tipo_veh ----------------------------------------------
top_veh = [1, 2, 3, 4, 5]
labels = {1:'Automóvil', 2:'Camioneta', 3:'Pick up', 4:'Moto', 5:'Camión'}

cross2 = pd.crosstab(
    df_fl[df_fl['tipo_veh'].isin(top_veh)]['tipo_veh'],
    df_fl[df_fl['tipo_veh'].isin(top_veh)]['fall_les'].map({1:'Fallecido', 2:'Lesionado'}),
    normalize='index'
).round(3) * 100

cross2.index = [labels[i] for i in cross2.index]
print(cross2.to_string())

fall_les   Fallecido  Lesionado
Automóvil       21.3       78.7
Camioneta       17.2       82.8
Pick up         17.4       82.6
Moto            18.2       81.8
Camión          27.8       72.2


### fall_les vs tipo_veh : Tasa de mortalidad por vehículo

| Vehículo | Fallecido | Lesionado |
|---|---|---|
| Automóvil | 21.3% | 78.7% |
| Camioneta | 17.2% | 82.8% |
| Pick up | 17.4% | 82.6% |
| Moto | 18.2% | 81.8% |
| Camión | **27.8%** | 72.2% |

**Camión es el más letal (27.8%)** : masa y velocidad en carretera.  
Moto no es el más letal por vehículo (18.2%) pero sí el más frecuente —  
su volumen lo convierte en el mayor contribuidor absoluto de fallecidos.  
Camioneta y Pick up tienen tasas similares (~17%).

In [11]:
# -- fall_les vs g_hora_5 ----------------------------------------------
cross3 = pd.crosstab(
    df_fl[df_fl['g_hora_5'] != 4]['g_hora_5'],
    df_fl[df_fl['g_hora_5'] != 4]['fall_les'].map({1:'Fallecido', 2:'Lesionado'}),
    normalize='index'
).round(3) * 100

cross3.index = ['Mañana', 'Tarde', 'Noche']
print(cross3.to_string())

fall_les  Fallecido  Lesionado
Mañana         21.2       78.8
Tarde          14.9       85.1
Noche          20.3       79.7


### fall_les vs g_hora_5 : Tasa de mortalidad por franja horaria

| Franja | Fallecido | Lesionado |
|---|---|---|
| Mañana | **21.2%** | 78.8% |
| Tarde | 14.9% | 85.1% |
| Noche | 20.3% | 79.7% |

**Tarde es la franja más segura (14.9% mortalidad)** : mayor tráfico, menor velocidad.  
Mañana es la más letal (21.2%) : posible mayor velocidad en vías con menos tráfico.  
Noche y Mañana tienen tasas similares (~20–21%).  
`g_hora_5` aporta señal predictiva moderada para el modelo.

In [12]:
# -- fall_les vs sexo_per ----------------------------------------------
cross4 = pd.crosstab(
    df_fl[df_fl['sexo_per'] != 9]['sexo_per'],
    df_fl[df_fl['sexo_per'] != 9]['fall_les'].map({1:'Fallecido', 2:'Lesionado'}),
    normalize='index'
).round(3) * 100

cross4.index = ['Hombre', 'Mujer']
print(cross4.to_string())

fall_les  Fallecido  Lesionado
Hombre         22.1       77.9
Mujer          10.5       89.5


### fall_les vs sexo_per : Tasa de mortalidad por sexo

| Sexo | Fallecido | Lesionado |
|---|---|---|
| Hombre | **22.1%** | 77.9% |
| Mujer | 10.5% | 89.5% |

**Hombres tienen el doble de tasa de mortalidad que mujeres (22.1% vs 10.5%).**  
Probable explicación: hombres predominan en moto, pick up y camión —  
vehículos de mayor exposición a eventos letales.  
`sexo_per` es variable predictora relevante para el modelo.

In [13]:
# -- Resumen ejecutivo fallecidos/lesionados ----------------------------------------------
resumen = {
    'Total registros'       : f'{len(df_fl):,}',
    'Fallecidos'            : f'{(df_fl["fall_les"]==1).sum():,} (19.1%)',
    'Lesionados'            : f'{(df_fl["fall_les"]==2).sum():,} (80.9%)',
    'Variable target'       : 'fall_les — 0% ignorados',
    'Predictores fuertes'   : 'tipo_eve, tipo_veh, sexo_per, g_hora_5',
    'Desbalance clases'     : 'Ratio 4.2:1 — requiere balanceo',
    'Hallazgo crítico'      : 'Código 9 en fall_les (9 registros) — eliminar',
    'Acción ETL'            : 'Normalizar nombres + eliminar zona_ciudad + drop código 9',
}

print(f'{"Campo":<25} {"Valor"}')
print('-' * 65)
for k, v in resumen.items():
    print(f'{k:<25} {v}')

Campo                     Valor
-----------------------------------------------------------------
Total registros           71,951
Fallecidos                13,741 (19.1%)
Lesionados                58,201 (80.9%)
Variable target           fall_les — 0% ignorados
Predictores fuertes       tipo_eve, tipo_veh, sexo_per, g_hora_5
Desbalance clases         Ratio 4.2:1 — requiere balanceo
Hallazgo crítico          Código 9 en fall_les (9 registros) — eliminar
Acción ETL                Normalizar nombres + eliminar zona_ciudad + drop código 9


### Resumen Ejecutivo : Fallecidos y Lesionados 2018–2024

**Dataset:** 71,951 registros : el más limpio de los tres

**Variable target:** `fall_les` : 0% ignorados, lista para el modelo
- Fallecido: 13,741 (19.1%)
- Lesionado: 58,201 (80.9%)
- Desbalance 4.2:1 → requiere técnicas de balanceo durante el entrenamiento.

**Predictores identificados por tasa de mortalidad:**
- `tipo_eve` — Derrape (30.1%) y Atropello (26.5%) más letales
- `tipo_veh` — Camión (27.8%) más letal, Moto mayor volumen absoluto
- `sexo_per` — Hombre duplica mortalidad vs Mujer (22.1% vs 10.5%)
- `g_hora_5` — Tarde más segura (14.9%), Mañana más letal (21.2%)

**Pendiente ETL:**
1. Normalizar nombres de columnas
2. Eliminar `zona_ciudad` (2021)
3. Eliminar 9 registros con `fall_les` = 9

---
**Resumen global de los 3 datasets:**

| Dataset | Registros | Calidad |
|---|---|---|
| Hechos | 52,488 | Buena |
| Vehículos involucrados | 80,721 | Buena |
| Fallecidos/Lesionados | 71,951 | Muy buena |